# Phase 1 — PyTorch Basics

This notebook covers the core PyTorch concepts you need before touching any face detection code.

Since you already know TensorFlow, every section includes a **TF → PyTorch** comparison so you can map what you already know.

**What we cover:**
1. Tensors — creating, shaping, moving to GPU
2. Autograd — automatic differentiation
3. `nn.Module` — building neural network layers
4. A complete training loop — forward pass → loss → backward → update
5. Saving and loading models

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

# We will use this throughout — always move tensors to the right device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device    : {DEVICE}')

---
## 1. Tensors

PyTorch tensors work almost identically to TensorFlow tensors, with one key difference:
**PyTorch tensors are mutable by default** (like numpy arrays), while TF tensors are immutable.

```
TensorFlow               PyTorch
──────────────────────   ──────────────────────
tf.zeros([3, 4])      →  torch.zeros(3, 4)
tf.ones([3, 4])       →  torch.ones(3, 4)
tf.constant([1,2,3])  →  torch.tensor([1,2,3])
tf.random.normal([])  →  torch.randn()
tensor.numpy()        →  tensor.numpy()          (same!)
tensor.shape          →  tensor.shape             (same!)
```

In [ ]:
# --- Creating tensors ---

# From a Python list
t1 = torch.tensor([1.0, 2.0, 3.0])
print('From list     :', t1)

# Zeros and ones
t2 = torch.zeros(3, 4)   # shape (3, 4) filled with 0s
t3 = torch.ones(2, 3)    # shape (2, 3) filled with 1s
print('Zeros shape   :', t2.shape)
print('Ones shape    :', t3.shape)

# Random — normal distribution (mean=0, std=1)
t4 = torch.randn(3, 3)
print('\nRandom 3x3:\n', t4)

# From numpy
arr = np.array([10, 20, 30], dtype=np.float32)
t5 = torch.from_numpy(arr)
print('\nFrom numpy    :', t5)

In [ ]:
# --- Tensor dtypes ---
# Most common in deep learning: float32 (default), int64, bool

a = torch.tensor([1.0, 2.0])          # float32 by default
b = torch.tensor([1, 2], dtype=torch.float32)  # explicit
c = torch.tensor([True, False])        # bool

print('float32:', a.dtype)
print('explicit float32:', b.dtype)
print('bool:', c.dtype)

# Cast dtype
d = a.to(torch.float64)
print('cast to float64:', d.dtype)

In [ ]:
# --- Moving tensors to GPU ---
# In TF this is automatic. In PyTorch you are explicit about it.
# This gives you more control — important in detection models.

x = torch.randn(3, 3)
print('On CPU:', x.device)

x_gpu = x.to(DEVICE)   # moves to CUDA if available
print('On GPU:', x_gpu.device)

# Best practice: always use DEVICE variable, not hardcode 'cuda'
# so the code works on CPU machines too

In [ ]:
# --- Tensor operations (same as numpy / TF) ---

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print('Add            :\n', a + b)
print('Element mul    :\n', a * b)
print('Matrix mul     :\n', a @ b)        # same as tf.matmul
print('Transpose      :\n', a.T)
print('Sum all        :', a.sum())
print('Mean           :', a.mean())
print('Reshape        :', a.reshape(1, 4))  # like tf.reshape
print('View (no copy) :', a.view(4))        # PyTorch-specific, faster

---
## 2. Autograd — automatic differentiation

PyTorch tracks operations on tensors with `requires_grad=True` and computes gradients via `.backward()`.

```
TensorFlow                      PyTorch
──────────────────────────────  ──────────────────────────────────
with tf.GradientTape() as t:    # No context manager needed!
    y = x * x                   y = x * x
grad = t.gradient(y, x)         y.backward()
                                grad = x.grad
```

PyTorch builds the computation graph **dynamically** as you run code (eager by default, just like TF2).

In [ ]:
# Scalar example: y = x^2, dy/dx = 2x

x = torch.tensor(3.0, requires_grad=True)  # tell PyTorch to track this
y = x ** 2

y.backward()   # compute dy/dx

print(f'x      = {x.item()}')
print(f'y      = {y.item()}')
print(f'dy/dx  = {x.grad.item()}')   # should be 2*3 = 6

In [ ]:
# More realistic: a tiny forward pass
# z = sum( (w * x + b)^2 )

x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([0.5, -0.3, 0.8], requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)

output = (w * x + b).pow(2).sum()
output.backward()

print('w.grad:', w.grad)   # dz/dw for each weight
print('b.grad:', b.grad)   # dz/db

# Note: gradients ACCUMULATE in PyTorch!
# You must zero them before each backward pass in a training loop.
# (We will see this in Section 4)

In [ ]:
# torch.no_grad() — disables gradient tracking for inference
# Same as TF's @tf.function or just running outside GradientTape

x = torch.tensor([1.0, 2.0], requires_grad=True)

with torch.no_grad():
    y = x * 2   # no graph built here

print('requires_grad during inference:', y.requires_grad)  # False
# Always use this during validation / inference — saves memory and compute

---
## 3. Building models with `nn.Module`

```
TensorFlow (Keras)              PyTorch
──────────────────────────────  ──────────────────────────────────
class M(tf.keras.Model):        class M(nn.Module):
    def __init__(self):             def __init__(self):
        super().__init__()              super().__init__()
        self.fc = Dense(10)             self.fc = nn.Linear(in, 10)

    def call(self, x):              def forward(self, x):
        return self.fc(x)               return self.fc(x)
```

Key difference: in PyTorch the method is `forward()`, not `call()`.

In [ ]:
# A simple fully-connected network

class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # nn.Linear = Dense layer  (in_features, out_features)
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        # This is called when you do model(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleNet(input_size=784, hidden_size=128, output_size=10)
model = model.to(DEVICE)   # move model weights to GPU

print(model)
print('\nTotal parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
# Common layers you will use in this project

# Linear (Dense in TF)
linear = nn.Linear(128, 64)

# Conv2d  — the core of any CNN / face detector
# (in_channels, out_channels, kernel_size, stride, padding)
conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)

# BatchNorm2d (after conv layers)
bn = nn.BatchNorm2d(32)

# MaxPool2d (reduce spatial size)
pool = nn.MaxPool2d(kernel_size=2, stride=2)

# Dropout
dropout = nn.Dropout(p=0.5)

# Activations
relu    = nn.ReLU()
sigmoid = nn.Sigmoid()

# Quick test: pass a fake image batch through conv → bn → relu → pool
x = torch.randn(4, 3, 64, 64)   # (batch=4, channels=3, H=64, W=64)
out = pool(relu(bn(conv(x))))
print('Input  :', x.shape)
print('Output :', out.shape)     # spatial dims halved by pool

In [ ]:
# nn.Sequential — stack layers without writing forward()
# Like tf.keras.Sequential

block = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
)

x = torch.randn(1, 3, 64, 64)
print('Sequential output shape:', block(x).shape)

---
## 4. The Training Loop

This is the biggest conceptual shift from Keras to PyTorch.

Keras: `model.compile()` + `model.fit()` — everything is hidden.

PyTorch: **you write the loop yourself** — more code, but you see exactly what's happening at every step. This is very useful when building custom things like face detectors.

The loop structure is always:
```
for each batch:
    1. optimizer.zero_grad()   # clear gradients from last step
    2. output = model(input)   # forward pass
    3. loss = criterion(output, target)  # compute loss
    4. loss.backward()         # compute gradients
    5. optimizer.step()        # update weights
```

In [ ]:
# Full training loop example — fitting y = 2x + 1 with a linear model
# (The simplest possible real training run)

torch.manual_seed(42)

# --- Fake dataset ---
X = torch.randn(200, 1).to(DEVICE)
y = 2 * X + 1 + 0.1 * torch.randn(200, 1).to(DEVICE)  # y = 2x + 1 + noise

# --- Model: single linear layer (no activation — regression) ---
model = nn.Linear(1, 1).to(DEVICE)

# --- Loss function ---
# MSELoss = mean squared error  (good for regression)
# CrossEntropyLoss              (for classification, later)
criterion = nn.MSELoss()

# --- Optimizer ---
# Adam is the go-to for most deep learning tasks
# SGD is also common for CNNs with momentum
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- Training loop ---
losses = []

for epoch in range(200):
    # Step 1: zero gradients — ALWAYS do this first
    # PyTorch accumulates gradients; if you forget this, weights diverge
    optimizer.zero_grad()

    # Step 2: forward pass
    predictions = model(X)

    # Step 3: compute loss
    loss = criterion(predictions, y)

    # Step 4: backward pass — compute all gradients
    loss.backward()

    # Step 5: update weights
    optimizer.step()

    losses.append(loss.item())   # .item() extracts Python float from tensor

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss.item():.4f}')

# Learned parameters (should be close to w=2, b=1)
w, b = model.weight.item(), model.bias.item()
print(f'\nLearned: y = {w:.3f}x + {b:.3f}')
print(f'Target : y = 2.000x + 1.000')

In [ ]:
# Plot the loss curve — should drop and flatten
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title('Training loss')
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Train / Val split + `model.train()` vs `model.eval()`

This is an important PyTorch detail. Layers like **BatchNorm** and **Dropout** behave differently during training vs inference. You must switch modes explicitly:

- `model.train()` — enables dropout, updates batchnorm running stats
- `model.eval()` — freezes batchnorm, disables dropout

Forgetting `model.eval()` during validation is a common bug — your val metrics will be wrong.

In [ ]:
# Template for a proper train + validation loop
# You will use this exact pattern in every future phase

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()           # <-- training mode
    total_loss = 0.0

    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)   # average loss over all batches


def evaluate(model, loader, criterion, device):
    model.eval()            # <-- eval mode
    total_loss = 0.0

    with torch.no_grad():   # no gradients needed for validation
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()

    return total_loss / len(loader)


print('Functions defined. Pattern to use in every future notebook:')
print('  for epoch in range(NUM_EPOCHS):')
print('      train_loss = train_one_epoch(...)')
print('      val_loss   = evaluate(...)')

---
## 6. Saving and loading models

In [ ]:
import os

# Best practice: save the state_dict (weights only), not the whole model
# The whole model save ties you to a specific file structure — avoid it.

save_path = '../models/linear_test.pth'
os.makedirs('../models', exist_ok=True)

# Save
torch.save(model.state_dict(), save_path)
print(f'Model saved to {save_path}')

# Load
model_loaded = nn.Linear(1, 1).to(DEVICE)           # create architecture first
model_loaded.load_state_dict(torch.load(save_path))  # then load weights into it
model_loaded.eval()

# Verify it works
test_input = torch.tensor([[3.0]]).to(DEVICE)
with torch.no_grad():
    pred = model_loaded(test_input)
print(f'Prediction for x=3: {pred.item():.3f}  (expected ≈ {2*3+1})')

In [ ]:
# Saving a checkpoint (more complete — includes optimizer state + epoch)
# Use this when you want to resume training later

checkpoint = {
    'epoch': 200,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': losses[-1],
}
torch.save(checkpoint, '../models/checkpoint_test.pth')

# Loading a checkpoint
ckpt = torch.load('../models/checkpoint_test.pth')
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
start_epoch = ckpt['epoch']

print(f'Resumed from epoch {start_epoch}, last loss: {ckpt["loss"]:.4f}')

---
## Summary

| Concept | PyTorch | TensorFlow equivalent |
|---------|---------|----------------------|
| Tensor | `torch.tensor(...)` | `tf.constant(...)` |
| Move to GPU | `.to(device)` | automatic |
| Gradient tracking | `requires_grad=True` | inside `GradientTape` |
| Backprop | `loss.backward()` | `tape.gradient(loss, vars)` |
| Model base | `nn.Module` | `tf.keras.Model` |
| Forward method | `forward(self, x)` | `call(self, x)` |
| Dense layer | `nn.Linear(in, out)` | `Dense(out)` |
| Conv layer | `nn.Conv2d(in, out, k)` | `Conv2D(out, k)` |
| Optimizer | `torch.optim.Adam(...)` | `tf.keras.optimizers.Adam(...)` |
| Training mode | `model.train()` | automatic in `.fit()` |
| Inference mode | `model.eval()` + `no_grad()` | automatic in `.predict()` |
| Save weights | `torch.save(state_dict)` | `model.save_weights(path)` |

---

**Next → Phase 2**: Load the WIDER Face dataset, visualize bounding box annotations, and build a `DataLoader`.